In [1]:
from dotenv import load_dotenv
import json
import whisper
from pyannote.audio import Pipeline
from pyannote_whisper.utils import diarize_text
import os

In [2]:
file = 'data/audio_sample.mp3'

In [3]:
load_dotenv('openai.env')

True

In [8]:
pipeline = Pipeline.from_pretrained("pyannote/speaker-diarization-3.1",
                                    use_auth_token=os.getenv('PYANNOTE'))

config.yaml:   0%|          | 0.00/469 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/5.91M [00:00<?, ?B/s]

config.yaml:   0%|          | 0.00/399 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/26.6M [00:00<?, ?B/s]

config.yaml:   0%|          | 0.00/221 [00:00<?, ?B/s]

In [5]:
model = whisper.load_model("base")

In [6]:
asr_result = model.transcribe(file)

/Users/declanbradley/.pyenv/versions/3.11.8/lib/python3.11/site-packages/whisper/transcribe.py:126: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


In [9]:
diarization_result = pipeline(file, num_speakers=2)

/Users/declanbradley/.pyenv/versions/3.11.8/lib/python3.11/site-packages/pyannote/audio/models/blocks/pooling.py:104: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/native/ReduceOps.cpp:1839.)
  std = sequences.std(dim=-1, correction=1)
/Users/declanbradley/.pyenv/versions/3.11.8/lib/python3.11/site-packages/torchaudio/_backend/soundfile_backend.py:71: UserWarning: The MPEG_LAYER_III subtype is unknown to TorchAudio. As a result, the bits_per_sample attribute will be set to 0. If you are seeing this warning, please report by opening an issue on github (after checking for existing/closed ones). You may otherwise ignore this warning.
  warnings.warn(
/Users/declanbradley/.pyenv/versions/3.11.8/lib/python3.11/site-packages/torchaudio/_backend/soundfile_backend.py:71: UserWarning: The MPEG_LAYER_III subtype is

In [11]:
final_result = diarize_text(asr_result, diarization_result)

In [35]:
asr_result

{'text': " So that's perfect. Yeah, let's talk more about like When like do your families and your friends understand why you're doing like the data journalism works Rather than you know be a pure engineer I know a lot of computer science spirit They're like oh, I want to be a software engineer I mean, yeah, I was a little go-go, but you're doing some like journalism works I do your families and your first understand and that they support you Yeah, I think I think so Friends I friends. I don't know. I don't know how my friends understand. I was certainly enthusiastic about my work Sometimes their eyes glades over a little and I talk about it, but I understand that it's a nerdy field right I Family yes my grandma actually on my mom's side was a journalist Yeah, I didn't really know her. She sadly passed when I was relatively young Before I got involved in the reporting space But there's certainly a family history there of reporting she was British so she worked in the UK press So yeah, 

In [38]:
speaker_matches = {list(final_result[i])[0].start : list(final_result[i])[1] for i in range(0, len(final_result))}

In [44]:
switches = list(speaker_matches.keys())
for i in range(0, len(asr_result['segments'])):
    curr_start = asr_result['segments'][i]['start']
    for j in range(0, len(switches)):
        if curr_start < switches[j]:
            switch = switches[j-1]
            break
    asr_result['segments'][i]['speaker'] = speaker_matches[switch]

In [47]:
result = asr_result

In [48]:
with open('transcript.json', 'w') as out_path:  
    json.dump(result, out_path)